# P105 — Product Cannibalization & Demand Shift Analysis
## Phase 3: Exploratory Data Analysis & Visualization

**Objective:** Visualize key patterns, trends, and before/after launch comparisons.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# Chart style
sns.set_theme(style='whitegrid', palette='Set2')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

# Create output folder
os.makedirs('../Images/Phase3', exist_ok=True)

LAUNCH_DATE = pd.Timestamp('2024-06-01')

## Load Cleaned Data

In [ ]:
df = pd.read_csv("../Data/cannibalization_cleaned.csv")
df['Date'] = pd.to_datetime(df['Date'])

print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

## 1. Distribution of Key Variables

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(['Sales', 'Price', 'Revenue', 'Rating', 'Marketing_Spend']):
    ax = axes[i]
    ax.hist(df[col], bins=40, color=sns.color_palette('Set2')[i], edgecolor='white')
    ax.set_title(f'{col}', fontweight='bold')
    ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():,.1f}')
    ax.legend(fontsize=8)

axes[5].axis('off')
plt.suptitle('Distribution of Key Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../Images/Phase3/01_distributions.png', bbox_inches='tight')
plt.show()

## 2. Correlation Heatmap

In [ ]:
num_cols = ['Price', 'Rating', 'Marketing_Spend', 'Sales', 'Revenue']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1)
plt.title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../Images/Phase3/02_correlation.png', bbox_inches='tight')
plt.show()

## 3. Monthly Sales Trend (with Launch Date)

In [ ]:
monthly = df.groupby('Year_Month').agg(
    Total_Sales   = ('Sales', 'sum'),
    Total_Revenue = ('Revenue', 'sum')
).reset_index()
monthly['Date'] = pd.to_datetime(monthly['Year_Month'])
monthly.sort_values('Date', inplace=True)

plt.figure(figsize=(14, 5))
plt.plot(monthly['Date'], monthly['Total_Sales'], marker='o', linewidth=2, label='Monthly Sales')
plt.axvline(LAUNCH_DATE, color='red', linestyle='--', linewidth=2, label='Launch Date (Jun 2024)')
plt.fill_betweenx([monthly['Total_Sales'].min(), monthly['Total_Sales'].max()],
                  monthly['Date'].min(), LAUNCH_DATE, alpha=0.05, color='blue', label='Before Launch')
plt.fill_betweenx([monthly['Total_Sales'].min(), monthly['Total_Sales'].max()],
                  LAUNCH_DATE, monthly['Date'].max(), alpha=0.05, color='green', label='After Launch')
plt.title('Monthly Sales Trend — Before & After Launch', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Total Sales (Units)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../Images/Phase3/03_monthly_sales_trend.png', bbox_inches='tight')
plt.show()

## 4. Monthly Revenue Trend

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(monthly['Date'], monthly['Total_Revenue'], marker='s', linewidth=2, color='green', label='Monthly Revenue')
plt.axvline(LAUNCH_DATE, color='red', linestyle='--', linewidth=2, label='Launch Date')
plt.title('Monthly Revenue Trend', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Total Revenue')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../Images/Phase3/04_monthly_revenue_trend.png', bbox_inches='tight')
plt.show()

## 5. Sales Trend by Product Group

In [ ]:
group_monthly = df.groupby(['Year_Month', 'Product_Group'])['Sales'].sum().reset_index()
group_monthly['Date'] = pd.to_datetime(group_monthly['Year_Month'])

fig, ax = plt.subplots(figsize=(15, 6))
for grp in sorted(df['Product_Group'].unique()):
    gdf = group_monthly[group_monthly['Product_Group'] == grp].sort_values('Date')
    lw = 2.5 if grp in ['G1', 'G2'] else 1
    ls = '-' if grp in ['G1', 'G2'] else '--'
    ax.plot(gdf['Date'], gdf['Sales'], linewidth=lw, linestyle=ls, label=grp)

ax.axvline(LAUNCH_DATE, color='red', linestyle='--', linewidth=2, label='Launch')
ax.set_title('Monthly Sales by Product Group (G1 & G2 = affected groups)', fontweight='bold')
ax.legend(ncol=5, fontsize=8)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../Images/Phase3/05_sales_by_group.png', bbox_inches='tight')
plt.show()

## 6. Category Sales — Before vs After Launch

In [ ]:
cat_period = df.groupby(['Category', 'Period_Flag'])['Sales'].mean().unstack()

cat_period.plot(kind='bar', figsize=(10, 5), edgecolor='white')
plt.title('Average Sales by Category — Before vs After Launch', fontweight='bold')
plt.ylabel('Avg Sales (Units)')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Period')
plt.tight_layout()
plt.savefig('../Images/Phase3/06_category_before_after.png', bbox_inches='tight')
plt.show()

## 7. Before vs After — Distribution Comparison

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for i, col in enumerate(['Sales', 'Revenue', 'Price']):
    sns.boxplot(data=df, x='Period_Flag', y=col, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{col}', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Before vs After Launch — Distribution Comparison', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/Phase3/07_before_after_boxplots.png', bbox_inches='tight')
plt.show()

## 8. Seasonality Analysis

In [ ]:
month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Monthly avg sales
monthly_avg = df.groupby('Month_Name')['Sales'].mean().reindex(month_order)
ax1.bar(range(12), monthly_avg.values, color=sns.color_palette('Set2'))
ax1.set_xticks(range(12))
ax1.set_xticklabels(month_order, rotation=45)
ax1.set_title('Average Sales by Month', fontweight='bold')
ax1.set_ylabel('Avg Sales')

# Quarterly
quarterly = df.groupby(['Year', 'Quarter'])['Sales'].sum().reset_index()
quarterly['Label'] = quarterly['Year'].astype(str) + '-Q' + quarterly['Quarter'].astype(str)
ax2.bar(range(len(quarterly)), quarterly['Sales'], color=sns.color_palette('Set2'))
ax2.set_xticks(range(len(quarterly)))
ax2.set_xticklabels(quarterly['Label'], rotation=45, ha='right')
ax2.set_title('Total Sales by Quarter', fontweight='bold')
ax2.set_ylabel('Total Sales')

plt.tight_layout()
plt.savefig('../Images/Phase3/08_seasonality.png', bbox_inches='tight')
plt.show()

## 9. Outlier Detection (IQR Method)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 5))

for i, col in enumerate(['Sales', 'Revenue', 'Price', 'Marketing_Spend']):
    sns.boxplot(data=df, y=col, ax=axes[i], color=sns.color_palette('Set2')[i])
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    axes[i].set_title(f'{col}\n({len(outliers)} outliers)', fontweight='bold', fontsize=10)

plt.suptitle('Outlier Detection — IQR Method', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../Images/Phase3/09_outlier_detection.png', bbox_inches='tight')
plt.show()

## 10. Demand Shift — P6 (Cannibalized) vs P4/P5 (New Launches)

In [ ]:
prod_monthly = df.groupby(['Year_Month', 'Product_ID'])['Sales'].sum().reset_index()
prod_monthly['Date'] = pd.to_datetime(prod_monthly['Year_Month'])

plt.figure(figsize=(14, 5))
for pid, color, label in [('P6', 'red', 'P6 (Cannibalized)'),
                           ('P4', 'blue', 'P4 (New Launch)'),
                           ('P5', 'green', 'P5 (New Launch)')]:
    pdata = prod_monthly[prod_monthly['Product_ID'] == pid].sort_values('Date')
    plt.plot(pdata['Date'], pdata['Sales'], marker='o', linewidth=2, color=color, label=label)

plt.axvline(LAUNCH_DATE, color='gray', linestyle='--', linewidth=2, label='Launch Date')
plt.title('Demand Shift — P6 Decline vs P4/P5 Rise (Group G2)', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Sales (Units)')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../Images/Phase3/10_demand_shift.png', bbox_inches='tight')
plt.show()

## 11. Region-wise Sales — Before vs After

In [ ]:
region_period = df.groupby(['Region', 'Period_Flag'])['Sales'].mean().unstack()

region_period.plot(kind='bar', figsize=(10, 5), edgecolor='white')
plt.title('Average Sales by Region — Before vs After Launch', fontweight='bold')
plt.ylabel('Avg Sales')
plt.xticks(rotation=0)
plt.legend(title='Period')
plt.tight_layout()
plt.savefig('../Images/Phase3/11_region_before_after.png', bbox_inches='tight')
plt.show()

## Phase 3 Summary

**Key Observations:**
- Sales and Revenue show relatively stable patterns with no extreme shifts
- P6 shows visible sales decline after launch date (Jun 2024)
- P4 and P5 appear only after launch (new products)
- Groups G1 and G2 are the affected groups containing launched products
- No major seasonality patterns detected
- Regional performance is relatively balanced across all 4 regions

**Next:** Phase 4 — Product Performance Analysis